# COCO Karpathy Test 5K: CLIP vs SigLIP Feature Encoding & Benchmark

This notebook performs:
1. GPU-accelerated batch feature extraction for 5,000 test images and 25,000 captions using `openai/clip-vit-base-patch32` and `google/siglip-base-patch16-224`.
2. Exact FAISS `IndexFlatIP` vector store construction with rich metadata.
3. Quantitative cross-modal evaluation (Recall@1, 5, 10, MRR, Modality Gap, Separation Margin).
4. Artifact export for downstream semantic search.

In [ ]:
!pip install -q faiss-cpu transformers

import os
import sys
import time
import json
import zipfile
from pathlib import Path

import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import faiss
from transformers import AutoProcessor, AutoModel, CLIPProcessor, CLIPModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB')


In [ ]:
# Locate coco_test_5k.json and images
search_roots = [Path('/kaggle/input'), Path('../datasets'), Path('./datasets'), Path('./')]
anno_path = None
for r in search_roots:
    matches = list(r.glob('**/coco_test_5k.json'))
    if matches:
        anno_path = matches[0]
        break

if not anno_path:
    raise FileNotFoundError('coco_test_5k.json not found in search paths!')

print(f'Annotation JSON located at: {anno_path}')
with open(anno_path, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

images_meta = coco_data['images']
print(f'Total images in split: {len(images_meta)}')

# Find val2014 image directory
img_dir = anno_path.parent.parent / 'val2014'
if not img_dir.exists():
    # Search for first image
    first_name = images_meta[0]['filename']
    for r in search_roots:
        found = list(r.glob(f'**/{first_name}'))
        if found:
            img_dir = found[0].parent
            break

print(f'Images directory located at: {img_dir}')
sample_exists = (img_dir / images_meta[0]['filename']).exists()
print(f'Sample image {images_meta[0]["filename"]} exists: {sample_exists}')


In [ ]:
def extract_tensor_features(output):
    if isinstance(output, torch.Tensor):
        return output
    if hasattr(output, 'pooler_output') and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, 'last_hidden_state') and output.last_hidden_state is not None:
        return output.last_hidden_state[:, 0, :]
    if hasattr(output, '__getitem__'):
        return output[0]
    return output

# Build linear list of images and captions
image_records = []
caption_records = []

for idx, img in enumerate(images_meta):
    filename = img['filename']
    full_path = img_dir / filename
    cocoid = img['cocoid']
    raw_captions = [s['raw'].strip() for s in img.get('sentences', [])]
    
    image_records.append({
        'idx': idx,
        'cocoid': cocoid,
        'filename': filename,
        'path': str(full_path),
        'captions': raw_captions,
    })
    
    for cap_idx, cap_text in enumerate(raw_captions):
        caption_records.append({
            'caption_idx': len(caption_records),
            'image_idx': idx,
            'cocoid': cocoid,
            'caption': cap_text,
        })

print(f'Indexed {len(image_records)} images and {len(caption_records)} captions.')


In [ ]:
# Load CLIP ViT-B/32
print('--> Loading openai/clip-vit-base-patch32...')
clip_model_id = 'openai/clip-vit-base-patch32'
clip_processor = CLIPProcessor.from_pretrained(clip_model_id)
clip_model = CLIPModel.from_pretrained(clip_model_id).to(device).eval()

# 1. Encode Images
batch_size = 128
clip_img_embeds = []
t0 = time.time()
with torch.no_grad():
    for i in tqdm(range(0, len(image_records), batch_size), desc='CLIP Images'):
        batch = image_records[i : i + batch_size]
        pil_imgs = [Image.open(b['path']).convert('RGB') for b in batch]
        inputs = clip_processor(images=pil_imgs, return_tensors='pt').to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            feats = extract_tensor_features(clip_model.get_image_features(**inputs))
            feats = feats / feats.norm(dim=-1, keepdim=True)
        clip_img_embeds.append(feats.cpu().float())

clip_img_tensor = torch.cat(clip_img_embeds, dim=0)
clip_img_time = time.time() - t0
print(f'CLIP Image Encoding: {len(clip_img_tensor)} images in {clip_img_time:.2f}s ({len(clip_img_tensor)/clip_img_time:.1f} img/s)')

# 2. Encode Captions
clip_txt_embeds = []
t0 = time.time()
with torch.no_grad():
    for i in tqdm(range(0, len(caption_records), 256), desc='CLIP Texts'):
        batch = caption_records[i : i + 256]
        texts = [b['caption'] for b in batch]
        inputs = clip_processor(text=texts, padding=True, truncation=True, return_tensors='pt').to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            feats = extract_tensor_features(clip_model.get_text_features(**inputs))
            feats = feats / feats.norm(dim=-1, keepdim=True)
        clip_txt_embeds.append(feats.cpu().float())

clip_txt_tensor = torch.cat(clip_txt_embeds, dim=0)
clip_txt_time = time.time() - t0
print(f'CLIP Text Encoding: {len(clip_txt_tensor)} captions in {clip_txt_time:.2f}s ({len(clip_txt_tensor)/clip_txt_time:.1f} txt/s)')

del clip_model, clip_processor
torch.cuda.empty_cache()


In [ ]:
# Load SigLIP ViT-B/16-224
print('--> Loading google/siglip-base-patch16-224...')
siglip_model_id = 'google/siglip-base-patch16-224'
siglip_processor = AutoProcessor.from_pretrained(siglip_model_id)
siglip_model = AutoModel.from_pretrained(siglip_model_id).to(device).eval()

# 1. Encode Images
siglip_img_embeds = []
t0 = time.time()
with torch.no_grad():
    for i in tqdm(range(0, len(image_records), batch_size), desc='SigLIP Images'):
        batch = image_records[i : i + batch_size]
        pil_imgs = [Image.open(b['path']).convert('RGB') for b in batch]
        inputs = siglip_processor(images=pil_imgs, return_tensors='pt').to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            feats = extract_tensor_features(siglip_model.get_image_features(**inputs))
            feats = feats / feats.norm(dim=-1, keepdim=True)
        siglip_img_embeds.append(feats.cpu().float())

siglip_img_tensor = torch.cat(siglip_img_embeds, dim=0)
siglip_img_time = time.time() - t0
print(f'SigLIP Image Encoding: {len(siglip_img_tensor)} images in {siglip_img_time:.2f}s ({len(siglip_img_tensor)/siglip_img_time:.1f} img/s)')

# 2. Encode Captions
siglip_txt_embeds = []
t0 = time.time()
with torch.no_grad():
    for i in tqdm(range(0, len(caption_records), 256), desc='SigLIP Texts'):
        batch = caption_records[i : i + 256]
        texts = [b['caption'] for b in batch]
        inputs = siglip_processor(text=texts, padding='max_length', truncation=True, return_tensors='pt').to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            feats = extract_tensor_features(siglip_model.get_text_features(**inputs))
            feats = feats / feats.norm(dim=-1, keepdim=True)
        siglip_txt_embeds.append(feats.cpu().float())

siglip_txt_tensor = torch.cat(siglip_txt_embeds, dim=0)
siglip_txt_time = time.time() - t0
print(f'SigLIP Text Encoding: {len(siglip_txt_tensor)} captions in {siglip_txt_time:.2f}s ({len(siglip_txt_tensor)/siglip_txt_time:.1f} txt/s)')

del siglip_model, siglip_processor
torch.cuda.empty_cache()


In [ ]:
def evaluate_coco(img_tensor, txt_tensor, image_records, caption_records, k_vals=[1, 5, 10]):
    print('Computing similarity matrix (25000 x 5000)...')
    # txt_tensor: (25000, D), img_tensor: (5000, D)
    sim_matrix = torch.matmul(txt_tensor, img_tensor.T) # shape: (25000, 5000)
    
    targets = torch.tensor([c['image_idx'] for c in caption_records], dtype=torch.long)
    N_queries = len(caption_records)
    
    # Text-to-Image (T2I)
    top_max = max(k_vals)
    _, topk_preds = sim_matrix.topk(top_max, dim=1, largest=True)
    target_expanded = targets.unsqueeze(1).expand_as(topk_preds)
    matches = (topk_preds == target_expanded)
    
    res = {}
    for k in k_vals:
        res[f'T2I_R@{k}'] = round(float((matches[:, :k].any(dim=1).float().mean().item()) * 100.0), 2)
    
    # MRR calculation
    _, full_sorted = sim_matrix.sort(dim=1, descending=True)
    ranks = (full_sorted == targets.unsqueeze(1)).nonzero(as_tuple=True)[1] + 1
    res['T2I_MRR'] = round(float((1.0 / ranks.float()).mean().item()), 4)
    res['T2I_mR'] = round(float(np.mean([res[f'T2I_R@{k}'] for k in k_vals])), 2)
    
    # Modality Gap: L2 distance between mean image embedding and mean text embedding
    mean_img = img_tensor.mean(dim=0)
    mean_txt = txt_tensor.mean(dim=0)
    gap = torch.norm(mean_img - mean_txt, p=2).item()
    res['modality_gap'] = round(float(gap), 4)
    
    # Separation Margin (mean positive score - mean top distractors)
    pos_scores = sim_matrix[torch.arange(N_queries), targets]
    res['mean_pos_score'] = round(float(pos_scores.mean().item()), 4)
    
    return res

print('\n=== EVALUATING CLIP (ViT-B/32) ON COCO 5K ===')
clip_benchmark = evaluate_coco(clip_img_tensor, clip_txt_tensor, image_records, caption_records)
clip_benchmark['img_enc_time_sec'] = round(clip_img_time, 2)
clip_benchmark['txt_enc_time_sec'] = round(clip_txt_time, 2)
print(json.dumps(clip_benchmark, indent=2))

print('\n=== EVALUATING SIGLIP (ViT-B/16-224) ON COCO 5K ===')
siglip_benchmark = evaluate_coco(siglip_img_tensor, siglip_txt_tensor, image_records, caption_records)
siglip_benchmark['img_enc_time_sec'] = round(siglip_img_time, 2)
siglip_benchmark['txt_enc_time_sec'] = round(siglip_txt_time, 2)
print(json.dumps(siglip_benchmark, indent=2))


In [ ]:
# Prepare storage directories
out_dir = Path('/kaggle/working')
storage_dir = out_dir / 'storage' / 'coco'
clip_store_dir = storage_dir / 'clip'
siglip_store_dir = storage_dir / 'siglip'
clip_store_dir.mkdir(parents=True, exist_ok=True)
siglip_store_dir.mkdir(parents=True, exist_ok=True)

# 1. Save raw numpy embeddings
np.save(out_dir / 'coco_clip_image_embeddings.npy', clip_img_tensor.numpy())
np.save(out_dir / 'coco_clip_text_embeddings.npy', clip_txt_tensor.numpy())
np.save(out_dir / 'coco_siglip_image_embeddings.npy', siglip_img_tensor.numpy())
np.save(out_dir / 'coco_siglip_text_embeddings.npy', siglip_txt_tensor.numpy())
print('Saved .npy embedding arrays.')

# 2. Build metadata list
metadata_list = []
for img in image_records:
    metadata_list.append({
        'id': f'coco_{img["idx"]}',
        'idx': img['idx'],
        'cocoid': img['cocoid'],
        'filename': img['filename'],
        'relative_path': f'val2014/{img["filename"]}',
        'captions': img['captions'],
    })

# 3. Build & Save CLIP FAISS Store
clip_index = faiss.IndexFlatIP(512)
clip_vecs = np.ascontiguousarray(clip_img_tensor.numpy(), dtype=np.float32)
faiss.normalize_L2(clip_vecs)
clip_index.add(clip_vecs)
faiss.write_index(clip_index, str(clip_store_dir / 'index.faiss'))

clip_meta_payload = {
    'dim': 512,
    'index_type': 'FlatIP',
    'model_name': 'clip',
    'count': len(metadata_list),
    'items': metadata_list,
}
with open(clip_store_dir / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(clip_meta_payload, f, ensure_ascii=False, indent=2)
print(f'Saved CLIP VectorStore ({clip_index.ntotal} vectors) -> {clip_store_dir}')

# 4. Build & Save SigLIP FAISS Store
siglip_index = faiss.IndexFlatIP(768)
siglip_vecs = np.ascontiguousarray(siglip_img_tensor.numpy(), dtype=np.float32)
faiss.normalize_L2(siglip_vecs)
siglip_index.add(siglip_vecs)
faiss.write_index(siglip_index, str(siglip_store_dir / 'index.faiss'))

siglip_meta_payload = {
    'dim': 768,
    'index_type': 'FlatIP',
    'model_name': 'siglip',
    'count': len(metadata_list),
    'items': metadata_list,
}
with open(siglip_store_dir / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(siglip_meta_payload, f, ensure_ascii=False, indent=2)
print(f'Saved SigLIP VectorStore ({siglip_index.ntotal} vectors) -> {siglip_store_dir}')

# 5. Save benchmark summary
full_benchmark = {
    'dataset': 'coco-karpathy-test5k',
    'total_images': len(image_records),
    'total_captions': len(caption_records),
    'clip_vit_b32': clip_benchmark,
    'siglip_vit_b16_224': siglip_benchmark,
}
with open(out_dir / 'coco_benchmark_results.json', 'w', encoding='utf-8') as f:
    json.dump(full_benchmark, f, indent=2)

# 6. Package compact ZIP for local download
zip_path = out_dir / 'coco_vector_stores.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(clip_store_dir / 'index.faiss', 'storage/coco/clip/index.faiss')
    zf.write(clip_store_dir / 'metadata.json', 'storage/coco/clip/metadata.json')
    zf.write(siglip_store_dir / 'index.faiss', 'storage/coco/siglip/index.faiss')
    zf.write(siglip_store_dir / 'metadata.json', 'storage/coco/siglip/metadata.json')
    zf.write(out_dir / 'coco_benchmark_results.json', 'coco_benchmark_results.json')

print(f'Created compact download archive ({zip_path.stat().st_size / (1024**2):.2f} MB) -> {zip_path}')
print('=== COCO INGESTION & BENCHMARK COMPLETE ===')
